In [1]:
import numpy as np
import pandas as pd
import optuna

from catboost import CatBoostClassifier

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.preprocessing import LabelEncoder

from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
df = pd.read_csv("train.csv")

print("Dataset shape:", df.shape)
display(df.head())
print(df["satisfaction"].value_counts(dropna=False))
print("Missing values:", df.isnull().sum().sum())

Dataset shape: (699635, 23)


,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False


satisfaction
False    389296
True     310339
Name: count, dtype: int64
Missing values: 292


In [4]:
def add_features(data):
    data = data.copy()

    # Flight delay features
    departure = data["Departure Delay in Minutes"]
    arrival = data["Arrival Delay in Minutes"]

    data["Total_Delay"] = (
        departure.fillna(0) + arrival.fillna(0)
    )

    data["Delay_Difference"] = arrival - departure

    data["Any_Delay"] = (
        (departure > 0) | (arrival > 0)
    ).astype(int)

    # Service-rating features
    service_cols = [
        "Inflight wifi service",
        "Departure/Arrival time convenient",
        "Ease of Online booking",
        "Gate location",
        "Food and drink",
        "Online boarding",
        "Seat comfort",
        "Inflight entertainment",
        "On-board service",
        "Leg room service",
        "Baggage handling",
        "Checkin service",
        "Cleanliness"
    ]

    data["Average_Service_Rating"] = (
        data[service_cols].mean(axis=1)
    )

    data["Low_Rating_Count"] = (
        data[service_cols].le(2).sum(axis=1)
    )

    return data

In [5]:
# Remove rows without a known target
df_model = df.dropna(subset=["satisfaction"]).copy()

# Keep IDs separate; never train on the identifier
ids = df_model["id"].copy() if "id" in df_model.columns else None

X = df_model.drop(
    columns=["id", "satisfaction"],
    errors="ignore"
).copy()

# Encode the target consistently
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(df_model["satisfaction"])

print("Target class mapping:")
for i, label in enumerate(target_encoder.classes_):
    print(f"{label} -> {i}")

# Check required feature columns
categorical_cols = [
    "Gender",
    "Type of Travel",
    "Class",
    "Customer Type"
]

service_cols = [
    "Inflight wifi service",
    "Departure/Arrival time convenient",
    "Ease of Online booking",
    "Gate location",
    "Food and drink",
    "Online boarding",
    "Seat comfort",
    "Inflight entertainment",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness"
]

required_cols = [
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
    *categorical_cols,
    *service_cols
]

missing_cols = [col for col in required_cols if col not in X.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# Split BEFORE tuning or feature selection
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Apply deterministic feature engineering to both partitions
X_train = add_features(X_train_raw)
X_test = add_features(X_test_raw)

# CatBoost requires categorical values to be strings
for col in categorical_cols:
    X_train[col] = X_train[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

Target class mapping:
False -> 0
True -> 1
Training shape: (559708, 26)
Test shape: (139927, 26)


In [7]:
baseline_model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    l2_leaf_reg=10,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1
)

baseline_model.fit(
    X_train,
    y_train,
    cat_features=categorical_cols
)

baseline_proba = baseline_model.predict_proba(X_test)[:, 1]

baseline_auc = roc_auc_score(y_test, baseline_proba)

print("Baseline test ROC-AUC:", round(baseline_auc, 5))

Baseline test ROC-AUC: 0.95726


In [8]:
def objective(trial):

    params = {
        "iterations": trial.suggest_categorical(
            "iterations", [600, 900, 1200, 1500]
        ),
        "depth": trial.suggest_int(
            "depth", 4, 8
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.015, 0.12, log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1.0, 30.0, log=True
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0.0, 2.0
        ),
        "border_count": trial.suggest_categorical(
            "border_count", [64, 128, 254]
        ),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": 42,
        "verbose": False,
        "allow_writing_files": False,
        "thread_count": -1
    }

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_train, y_train), start=1
    ):
        X_fold_train = X_train.iloc[train_idx]
        X_fold_valid = X_train.iloc[valid_idx]

        y_fold_train = y_train[train_idx]
        y_fold_valid = y_train[valid_idx]

        model = CatBoostClassifier(**params)

        model.fit(
            X_fold_train,
            y_fold_train,
            cat_features=categorical_cols
        )

        valid_proba = model.predict_proba(
            X_fold_valid
        )[:, 1]

        fold_auc = roc_auc_score(
            y_fold_valid,
            valid_proba
        )

        fold_scores.append(fold_auc)

    mean_auc = float(np.mean(fold_scores))
    std_auc = float(np.std(fold_scores))

    trial.set_user_attr("fold_scores", fold_scores)
    trial.set_user_attr("std_auc", std_auc)

    return mean_auc

In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="airline_catboost_roc_auc"
)

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

print("Best CV ROC-AUC:", round(study.best_value, 5))
print("\nBest hyperparameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

print("\nBest trial fold scores:")
print(study.best_trial.user_attrs["fold_scores"])

print(
    "Fold standard deviation:",
    round(study.best_trial.user_attrs["std_auc"], 5)
)

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
best_params = study.best_params.copy()

optuna_catboost_model = CatBoostClassifier(
    **best_params,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1
)

optuna_catboost_model.fit(
    X_train,
    y_train,
    cat_features=categorical_cols
)

print("Optuna-tuned CatBoost training complete.")
print("L2 regularization:", best_params["l2_leaf_reg"])
print("Trees trained:", optuna_catboost_model.tree_count_)

In [ ]:
test_proba = optuna_catboost_model.predict_proba(X_test)[:, 1]
test_pred = optuna_catboost_model.predict(X_test).ravel().astype(int)

test_auc = roc_auc_score(y_test, test_proba)

print("Baseline ROC-AUC:", round(baseline_auc, 5))
print("Optuna CV ROC-AUC:", round(study.best_value, 5))
print("Optuna test ROC-AUC:", round(test_auc, 5))
print("Change vs baseline:", round(test_auc - baseline_auc, 5))

print("\nOther test metrics:")
print("Accuracy:", round(accuracy_score(y_test, test_pred), 5))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 5))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 5))
print("F1-score:", round(f1_score(y_test, test_pred, zero_division=0), 5))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=[str(c) for c in target_encoder.classes_],
        zero_division=0
    )
)

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_pred))

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": optuna_catboost_model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

display(feature_importance.head(20))

In [ ]:
import matplotlib.pyplot as plt

top_features = feature_importance.head(20).sort_values("Importance")

plt.figure(figsize=(10, 8))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Feature Importance")
plt.title("Top 20 CatBoost Features")
plt.tight_layout()
plt.show()

In [ ]:
# Save model
optuna_catboost_model.save_model(
    "optuna_catboost_l2_features.cbm"
)

# Save test predictions
prediction_results = pd.DataFrame({
    "Actual_Satisfaction": target_encoder.inverse_transform(y_test),
    "Predicted_Satisfaction": target_encoder.inverse_transform(test_pred),
    "Probability_Class_0": optuna_catboost_model.predict_proba(X_test)[:, 0],
    "Probability_Class_1": test_proba
})

prediction_results.to_csv(
    "optuna_catboost_test_predictions.csv",
    index=False
)

# Save Optuna search results
study.trials_dataframe().to_csv(
    "optuna_trials.csv",
    index=False
)

print("Model and results saved successfully.")